# TRIBE v2 Demo — Mac / Apple Silicon Edition

This local Mac notebook now has conservative, report-backed modes for the MLX V-JEPA2 port. The default `safe` mode remains bounded and avoids long prediction work. `mlx_vjepa_probe` reads the generated MLX ViT-g probe and TRIBE smoke reports, while `mlx_vjepa_predict` refuses to run unless `TRIBEV2_MLX_VJEPA_PREDICT_OPT_IN=1` and all prior artifacts are present.

Compared with the Colab notebook, this version:

- removes the Colab runtime restart cell;
- bootstraps a **uv-managed Python 3.11 environment** and registers a Jupyter kernel;
- checks that the notebook is running with **Python 3.11+** as required by `pyproject.toml`;
- selects **MPS** when available, with CPU fallback enabled for unsupported PyTorch operations;
- verifies MLX availability on Apple Silicon;
- reports real MLX ViT-g V-JEPA2 probe status from `cache_mac_m2_verification/mlx_vjepa_vitg/metadata.json`;
- can run guarded TRIBE smoke predictions from precomputed MLX V-JEPA2 features;
- keeps heavyweight original extractor prediction paths opt-in.


## 0. Bootstrap the uv environment first

Run the next cell first. It creates a local uv virtual environment at `.venv-mac-py311/`, installs this checkout with plotting dependencies, and registers a Jupyter kernel named **TRIBE v2 Mac (uv Python 3.11)**.

If you are not already using that kernel, the cell will stop after setup. Select **Kernel → Change Kernel → TRIBE v2 Mac (uv Python 3.11)**, then rerun the notebook from the top.

This replaces the Colab/conda setup flow; no terminal setup is required beyond opening this notebook from the repository root.


In [ ]:
# UV environment bootstrap: run this cell first.
# It is safe to rerun; uv reuses the existing .venv-mac-py311 environment.
import os
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path

PROJECT_ROOT = Path.cwd()
VENV_DIR = PROJECT_ROOT / ".venv-mac-py311"
VENV_PYTHON = VENV_DIR / "bin" / "python"
KERNEL_NAME = "tribev2-mac"
KERNEL_DISPLAY_NAME = "TRIBE v2 Mac (uv Python 3.11)"

# Must be set before importing torch in later cells.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("MKL_NUM_THREADS", "8")


def run(cmd, *, check=True):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=PROJECT_ROOT, check=check, text=True)


def find_uv():
    candidates = [
        shutil.which("uv"),
        str(Path(sysconfig.get_path("scripts")) / "uv"),
        str(Path.home() / ".local" / "bin" / "uv"),
        "/opt/homebrew/bin/uv",
        "/usr/local/bin/uv",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return candidate
    return None

uv = find_uv()
if uv is None:
    print("uv was not found; installing uv into the current bootstrap interpreter...")
    run([sys.executable, "-m", "pip", "install", "-U", "uv"])
    uv = find_uv()
if uv is None:
    raise RuntimeError("Could not find uv after installation. Install uv manually, then rerun this cell.")
print(f"Using uv: {uv}")

# Create a Python 3.11 venv. If uv does not already have/find 3.11, ask uv to install it first.
if not VENV_PYTHON.exists():
    result = run([uv, "venv", "--python", "3.11", str(VENV_DIR)], check=False)
    if result.returncode != 0:
        run([uv, "python", "install", "3.11"])
        run([uv, "venv", "--python", "3.11", str(VENV_DIR)])
else:
    print(f"Reusing existing environment: {VENV_DIR}")

version = subprocess.check_output(
    [str(VENV_PYTHON), "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}')"],
    text=True,
).strip()
print(f"uv environment Python: {version} ({VENV_PYTHON})")
if tuple(map(int, version.split(".")[:2])) < (3, 11):
    raise RuntimeError(f"{VENV_DIR} is not Python 3.11+. Remove it and rerun this cell.")

# Install runtime, notebook kernel, and this checkout. uv will skip work when already satisfied.
run([uv, "pip", "install", "--python", str(VENV_PYTHON), "-U", "pip", "uv", "jupyter", "ipykernel", "mlx"])
run([uv, "pip", "install", "--python", str(VENV_PYTHON), "-e", ".[plotting]"])
# Install optional MLX-LM without allowing it to relax TRIBE's pinned numpy/click/typer bounds.
run([uv, "pip", "install", "--python", str(VENV_PYTHON), "--no-deps", "mlx-lm", "sentencepiece", "protobuf"])
run([
    str(VENV_PYTHON),
    "-m",
    "ipykernel",
    "install",
    "--user",
    "--name",
    KERNEL_NAME,
    "--display-name",
    KERNEL_DISPLAY_NAME,
])

current = Path(sys.executable).resolve()
target = VENV_PYTHON.resolve()
if current != target:
    print("\n✅ uv environment is ready, but this notebook is not currently running inside it.")
    print(f"Current kernel Python: {current}")
    print(f"Target kernel Python:  {target}")
    print(f"\nNext step: select Kernel → Change Kernel → {KERNEL_DISPLAY_NAME}, then rerun from the top.")
    raise RuntimeError("Switch to the uv-managed TRIBE v2 Mac kernel before continuing.")

# Make venv console scripts such as uv/uvx visible to subprocess calls made by TRIBE helpers.
os.environ["PATH"] = f"{VENV_PYTHON.parent}:{os.environ.get('PATH', '')}"
print("\n✅ Running inside the uv-managed TRIBE v2 Mac environment. Continue to the preflight cell.")


In [ ]:
# Runtime preflight: run this before installing/loading TRIBE v2.
# It prints the Mac hardware summary and enforces Python 3.11+.
import os
import platform
import subprocess
import shutil
import sys
from pathlib import Path

# Must be set before importing torch so unsupported MPS ops can fall back to CPU.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("MKL_NUM_THREADS", "8")
# Ensure uv/uvx and other venv console scripts are visible to subprocesses.
os.environ["PATH"] = f"{Path(sys.executable).parent}:{os.environ.get('PATH', '')}"

print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print(f"uv available: {shutil.which('uv') or 'not found'}")
print(f"uvx available: {shutil.which('uvx') or 'not found'}")
print(f"Platform: {platform.platform()}")

if platform.system() == "Darwin":
    def _cmd(*args):
        return subprocess.check_output(args, text=True).strip()

    try:
        cpu = _cmd("sysctl", "-n", "machdep.cpu.brand_string")
        mem_bytes = int(_cmd("sysctl", "-n", "hw.memsize"))
        cores = _cmd("sysctl", "-n", "hw.ncpu")
        macos = _cmd("sw_vers", "-productVersion")
        print(f"macOS: {macos}")
        print(f"CPU: {cpu}")
        print(f"Memory: {mem_bytes / 1024**3:.1f} GB")
        print(f"Logical cores: {cores}")
        if mem_bytes < 30 * 1024**3:
            print("WARNING: less than 32 GB memory detected; keep RUN_VIDEO_DEMO=False first.")
    except Exception as exc:
        print(f"Could not query full Mac hardware info: {exc}")

assert sys.version_info >= (3, 11), (
    "TRIBE v2 requires Python 3.11+. Create/select the tribev2-mac kernel above, "
    f"then rerun this notebook. Current Python is {sys.version.split()[0]}."
)


## Dependency sanity check

The uv bootstrap cell already installed TRIBE v2 and notebook dependencies into `.venv-mac-py311/`. This cell verifies that the active kernel can import the local checkout before loading the model.


In [ ]:
import sys
from pathlib import Path

import tribev2
import mlx.core as mx

print(f"Active Python: {sys.executable}")
print(f"TRIBE v2 import path: {Path(tribev2.__file__).resolve()}")
print(f"MLX default device: {mx.default_device()}")


## MLX vs MPS for this notebook

Apple Silicon has GPU acceleration through both **MLX** and PyTorch **MPS**. MLX is available and useful for MLX-native models, but TRIBE v2's released pipeline uses PyTorch/Hugging Face/neuralset extractors. A real MLX port would require converting/reimplementing the V-JEPA2, DINOv2, Wav2Vec-BERT, Llama-compatible text extractor, cache layout, and feature tensor compatibility.

So this notebook does two things:

1. verifies MLX can use the Apple GPU;
2. uses PyTorch MPS as the practical Apple-GPU acceleration path for the existing TRIBE feature extractors, with CPU fallback for unsupported ops.

Observed on this M2 Pro test run: MLX itself uses `Device(gpu, 0)`, and the notebook can patch TRIBE extractor objects to request PyTorch MPS. However, full V-JEPA2 video prediction remained impractically slow/hung even for a 1-second 224px smoke clip, so full prediction remains opt-in rather than the default.


In [ ]:
# MLX GPU feasibility check. This confirms MLX itself can see/use the Apple GPU.
import time
import mlx.core as mx

print(f"MLX default device: {mx.default_device()}")
a = mx.random.normal((1024, 1024))
b = mx.random.normal((1024, 1024))
t0 = time.perf_counter()
c = a @ b
mx.eval(c)
print(f"MLX matmul completed on {mx.default_device()} in {time.perf_counter() - t0:.3f}s; shape={c.shape}")


## MLX V-JEPA2 report-backed modes

The full MLX V-JEPA2 path is now represented by repo-local reports rather than vague backend claims.

- `safe`: default bounded notebook path.
- `mlx_vjepa_probe`: read and validate the MLX ViT-g probe plus c3/c1 TRIBE smoke reports.
- `mlx_vjepa_predict`: guarded prediction path using precomputed MLX V-JEPA2 features; it refuses to run unless `TRIBEV2_MLX_VJEPA_PREDICT_OPT_IN=1` and required artifacts exist.

Legacy modes (`mlx_probe`, `mlx_hybrid`, `mps_experimental`) are still accepted for older local workflows, but the V-JEPA2 status language below distinguishes tiny feasibility, real ViT-g success, and blockers.


In [ ]:
# Report-backed MLX V-JEPA2 status. This cell does not make false backend claims.
from pathlib import Path
import json
import os

NOTEBOOK_BACKEND_MODE = os.environ.get("TRIBEV2_NOTEBOOK_MODE", "safe")
RUN_FULL_MLX_BACKEND_ATTEMPT = NOTEBOOK_BACKEND_MODE == "mlx_probe"  # legacy incompatibility probe
RUN_MLX_VJEPA_REPORT = NOTEBOOK_BACKEND_MODE in {"mlx_vjepa_probe", "mlx_vjepa_predict"}
MLX_VJEPA_PROBE_REPORT = Path("cache_mac_m2_verification/mlx_vjepa_vitg/metadata.json")
MLX_VJEPA_C3_REPORT = Path("cache_mac_m2_verification/mlx_vjepa_tribe_c3.json")
MLX_VJEPA_C1_REPORT = Path("cache_mac_m2_verification/mlx_vjepa_tribe_c1_video_audio.json")
MLX_VJEPA_FEATURES = Path("cache_mac_m2_verification/mlx_vjepa_vitg/features.npy")

print(f"Notebook backend mode: {NOTEBOOK_BACKEND_MODE}")


def _load_report(path: Path):
    if not path.exists():
        return {"status": "missing", "path": str(path)}
    return json.loads(path.read_text())


if RUN_MLX_VJEPA_REPORT:
    reports = {
        "probe": _load_report(MLX_VJEPA_PROBE_REPORT),
        "c3_video": _load_report(MLX_VJEPA_C3_REPORT),
        "c1_video_audio": _load_report(MLX_VJEPA_C1_REPORT),
    }
    for name, report in reports.items():
        print(f"\n{name}: {report.get('status')}")
        if name == "probe":
            print("  feature_shape:", report.get("feature_shape"))
            print("  finite:", report.get("finite"))
            print("  parity:", (report.get("reference_parity") or {}).get("status"))
            print("  tiny/non-ViT-g substitute:", "not used for compatibility")
        else:
            print("  predictions_shape:", report.get("predictions_shape"))
            print("  predictions_finite:", report.get("predictions_finite"))
            print("  video_feature_shape:", report.get("video_feature_shape"))
            print("  video_blocker:", report.get("video_blocker"))
    required_ok = (
        reports["probe"].get("status") == "ok"
        and reports["probe"].get("finite") is True
        and reports["probe"].get("feature_shape", [])[:2] == [2, 1408]
    )
    print("\nMLX V-JEPA2 probe gate:", "passed" if required_ok else "blocked/missing")
elif RUN_FULL_MLX_BACKEND_ATTEMPT:
    import mlx.core as mx
    from huggingface_hub import list_models
    print("Legacy mlx_probe mode: documenting released-stack incompatibility, not claiming full MLX TRIBE.")
    print("MLX default device:", mx.default_device())
    for q in ["mlx vjepa2", "mlx dinov2", "mlx wav2vec bert", "mlx llama 3.2 3b"]:
        print(f"HF search: {q}")
        try:
            print([m.modelId for m in list(list_models(search=q, limit=3))])
        except Exception as exc:
            print("  search skipped:", exc)
else:
    print("MLX V-JEPA2 report read skipped in safe mode. Set TRIBEV2_NOTEBOOK_MODE=mlx_vjepa_probe to inspect reports.")


## Loading the model

The first run downloads the released TRIBE v2 checkpoint/config from Hugging Face (~1 GB) plus feature-extractor weights as needed. Subsequent runs reuse `cache_mac_m2/`.

Important backend notes:

- `DEVICE` controls the TRIBE brain model. On Apple Silicon it normally selects `mps`.
- `EXTRACTOR_DEVICE` starts as `cpu` on Apple Silicon because neuralset's pydantic validation does not accept `mps` or `mlx` extractor devices.
- `USE_MPS_FOR_EXTRACTORS_EXPERIMENTAL=True` then patches already-created extractor objects to request PyTorch MPS. This is the closest runnable Apple-GPU path for the released pipeline.
- A true full-MLX backend is not available without a port/reimplementation of TRIBE's PyTorch extractors and checkpoint handling.

If MPS runs into unsupported operations or memory pressure, set `PREFER_MPS=False` and rerun from this cell.

Backend mode flags are intentionally conservative: `BACKEND_MODE="safe"` validates events and probes only; `mlx_probe` documents MLX availability/blockers; `mps_experimental` requests PyTorch MPS for existing extractors; `mlx_hybrid` is reserved for repo-local adapters after contract tests pass.


In [ ]:
from pathlib import Path
import gc
import os

# Keep cache separate from the Colab/default notebook cache.
CACHE_FOLDER = Path("./cache_mac_m2")
CACHE_FOLDER.mkdir(exist_ok=True)

# Mac-safe defaults. Long prediction paths stay opt-in.
BACKEND_MODE = os.environ.get("TRIBEV2_NOTEBOOK_MODE", "safe")  # safe | mlx_vjepa_probe | mlx_vjepa_predict | mlx_probe | mps_experimental | mlx_hybrid
MLX_ADAPTERS = [
    item.strip()
    for item in os.environ.get("TRIBEV2_MLX_ADAPTERS", "").split(",")
    if item.strip()
]  # e.g. TRIBEV2_MLX_ADAPTERS=text_proof,text_mlx_lm after adapter contract tests pass
TEXT_MLX_LM_MODEL = os.environ.get("TRIBEV2_MLX_LM_MODEL", "mlx-community/Llama-3.2-3B-Instruct-4bit")
PREFER_MPS = os.environ.get("TRIBEV2_PREFER_MPS", "1") == "1"
USE_MPS_FOR_EXTRACTORS_EXPERIMENTAL = os.environ.get("TRIBEV2_USE_MPS_EXTRACTORS", "1") == "1"
RUN_TEXT_DEMO = False  # Keep the default run focused on the requested local video/no-text cases.
RUN_LOCAL_VIDEO_DEMO = True
RUN_LOCAL_VIDEO_PREDICTION = os.environ.get("TRIBEV2_RUN_LOCAL_VIDEO_PREDICTION", "0") == "1"  # Original V-JEPA/DINO extractor path remains opt-in.
RUN_MLX_VJEPA_PREDICT = BACKEND_MODE == "mlx_vjepa_predict"
MLX_VJEPA_PREDICT_OPT_IN = os.environ.get("TRIBEV2_MLX_VJEPA_PREDICT_OPT_IN", "0") == "1"
VIDEO_PREDICTION_TIMEOUT_SECONDS = int(os.environ.get("TRIBEV2_VIDEO_PREDICTION_TIMEOUT_SECONDS", "900"))
MAX_LOCAL_VIDEO_CASES_PER_FOLDER = 1  # Increase after the smoke test passes.
SMOKE_TEST_SECONDS = float(os.environ.get("TRIBEV2_SMOKE_TEST_SECONDS", "5.0"))
SMOKE_TEST_WIDTH = int(os.environ.get("TRIBEV2_SMOKE_TEST_WIDTH", "384"))

import torch

try:
    torch.set_num_threads(min(8, os.cpu_count() or 8))
except Exception:
    pass

def pick_device(prefer_mps: bool = True) -> str:
    if torch.cuda.is_available():
        return "cuda"
    if prefer_mps and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = pick_device(PREFER_MPS)
# TRIBE/neuralset feature extractors validate only auto/cpu/cuda/accelerate.
# On Apple Silicon, keep extractor models on CPU while allowing the TRIBE brain model on MPS.
EXTRACTOR_DEVICE = "cpu" if DEVICE == "mps" else DEVICE
print(f"Backend mode: {BACKEND_MODE}")
print(f"MLX adapters: {MLX_ADAPTERS or []}")
print(f"Prefer MPS: {PREFER_MPS}; experimental MPS extractors: {USE_MPS_FOR_EXTRACTORS_EXPERIMENTAL}")
print(f"Smoke clip: {SMOKE_TEST_SECONDS}s at width {SMOKE_TEST_WIDTH}")
print(f"MLX V-JEPA predict requested: {RUN_MLX_VJEPA_PREDICT}; opt-in: {MLX_VJEPA_PREDICT_OPT_IN}")
print(f"Using torch device for TRIBE brain model: {DEVICE}")
print(f"Using feature extractor device: {EXTRACTOR_DEVICE}")
print(f"MPS fallback enabled: {os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK')}")

if DEVICE == "mps":
    try:
        # Leave headroom for the OS/window server on unified memory.
        torch.mps.set_per_process_memory_fraction(0.85)
    except Exception as exc:
        print(f"Could not set MPS memory fraction: {exc}")
elif DEVICE == "cpu":
    print("Running on CPU. This is compatible but can be slow for full-model inference.")

from tribev2.demo_utils import TribeModel, download_file
from tribev2.plotting import PlotBrain

model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    device=DEVICE,
    config_update={
        # Public mirror of the gated Meta Llama 3.2 3B dependency used by
        # facebook/tribev2's released config. The mirror keeps the same
        # Llama hidden size/layer layout, so feature dimensions stay compatible
        # without requiring access approval for meta-llama/Llama-3.2-3B.
        "data.text_feature.model_name": "unsloth/Llama-3.2-3B",
        "data.text_feature.device": EXTRACTOR_DEVICE,
        "data.audio_feature.device": EXTRACTOR_DEVICE,
        "data.video_feature.image.device": EXTRACTOR_DEVICE,
        "data.image_feature.image.device": EXTRACTOR_DEVICE,
    },
)


def enable_experimental_mps_extractors(xp) -> bool:
    """Try to run TRIBE feature extractors on Apple GPU via PyTorch MPS.

    MLX is available on Apple Silicon, but TRIBE/neuralset feature extractors are
    implemented as PyTorch/Hugging Face modules. The practical acceleration path
    inside this notebook is therefore PyTorch MPS, not a full MLX rewrite.

    Pydantic validation in neuralset currently accepts only auto/cpu/cuda/accelerate,
    so the model is constructed with CPU extractors first and then the already-built
    extractor objects are switched to mps. If an extractor hits an unsupported op,
    PYTORCH_ENABLE_MPS_FALLBACK=1 allows CPU fallback for that op.
    """
    if DEVICE != "mps" or not USE_MPS_FOR_EXTRACTORS_EXPERIMENTAL:
        return False
    patched = []

    def force_device(obj, attr="device"):
        if obj is not None and hasattr(obj, attr):
            object.__setattr__(obj, attr, "mps")
            patched.append(f"{type(obj).__name__}.{attr}")

    for attr in ("text_feature", "audio_feature"):
        force_device(getattr(xp.data, attr, None))
    for attr in ("video_feature", "image_feature"):
        extractor = getattr(xp.data, attr, None)
        image = getattr(extractor, "image", None)
        force_device(image)
    print("Experimental MPS extractor patch applied:", ", ".join(patched) or "none")
    return bool(patched)


MPS_EXTRACTORS_ENABLED = enable_experimental_mps_extractors(model)
print(f"Experimental MPS extractors enabled: {MPS_EXTRACTORS_ENABLED}")
plotter = PlotBrain(mesh="fsaverage5")


## Mac-safe text demo: direct word events

The original notebook's text path converts text to speech with gTTS and then runs WhisperX to recover word timings. That is useful for full audio-style preprocessing, but it downloads/runs a large speech model and is slow on CPU-only WhisperX.

For a reliable first local Mac run, this cell creates simple word-level events directly. This exercises TRIBE v2 text features and brain prediction without the heavy transcription pipeline. If you specifically want the original gTTS/WhisperX path, use `model.get_events_dataframe(text_path=...)` after the main demo works.


In [ ]:
import re
import pandas as pd
from neuralset.events.utils import standardize_events

text = '''
To be or not to be, that is the question.
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die, to sleep,
No more; and by a sleep to say we end
The heartache and the thousand natural shocks
'''.strip()

def build_word_events(text: str, words_per_second: float = 2.4) -> pd.DataFrame:
    words = re.findall(r"[\w']+|[^\w\s]", text)
    rows = []
    t = 0.0
    step = 1.0 / words_per_second
    context = " ".join(words)
    for i, word in enumerate(words):
        duration = max(0.20, min(0.75, step * max(1.0, len(word) / 5)))
        rows.append(
            {
                "type": "Word",
                "text": word,
                "sentence": context,
                "context": context,
                "start": round(t, 3),
                "duration": round(duration, 3),
                "sequence_id": 0,
                "filepath": "mac_direct_text",
                "timeline": "default",
                "subject": "default",
            }
        )
        t += duration + 0.05
    return standardize_events(pd.DataFrame(rows))

if RUN_TEXT_DEMO:
    df = build_word_events(text)
    display(df.head(12)[["type", "start", "duration", "text", "context"]])
    print(f"Built {len(df)} word events spanning about {df.start.max() + df.duration.iloc[-1]:.1f} seconds")
else:
    print("RUN_TEXT_DEMO is False; skipping text event construction.")


### Run text prediction

This can still take time on first run because the Llama-compatible text extractor weights must be downloaded and cached.


In [ ]:
if RUN_TEXT_DEMO:
    preds, segments = model.predict(events=df)
    print(f"Predictions shape: {preds.shape}  (n_timesteps, n_vertices)")
else:
    print("RUN_TEXT_DEMO is False; skipping text prediction.")


### Visualize text predictions

For audio/text-only stimuli, the stimulus panel shows the active words for each time step.


In [ ]:
if RUN_TEXT_DEMO:
    n_timesteps = min(15, len(preds))
    fig = plotter.plot_timesteps(
        preds[:n_timesteps],
        segments=segments[:n_timesteps],
        cmap="fire",
        norm_percentile=99,
        vmin=.6,
        alpha_cmap=(0, .2),
        show_stimuli=True,
    )
else:
    print("RUN_TEXT_DEMO is False; skipping text visualization.")


## Optional MLX-assisted text adapters

This notebook now has two opt-in MLX text lanes:

- `text_proof`: fastest contract proof. It creates correctly-shaped deterministic MLX tensors and verifies the TRIBE PyTorch brain model can consume them.
- `text_mlx_lm`: closer semantic lane. It loads an MLX-LM Llama-3.2 3B 4-bit model on Apple GPU, extracts final hidden states for Word events, shapes them as TRIBE text features, and runs the released TRIBE brain model. This is still labeled **not exact neuralset parity** because neuralset's HuggingFaceText extractor aggregates selected intermediate layers while this lane uses MLX-LM final hidden states.

To run both in the notebook, set before launching the kernel or in the environment used by `tools/run_notebook_safe.py`:

```bash
TRIBEV2_NOTEBOOK_MODE=mlx_hybrid
TRIBEV2_MLX_ADAPTERS=text_proof,text_mlx_lm
# optional override:
TRIBEV2_MLX_LM_MODEL=mlx-community/Llama-3.2-3B-Instruct-4bit
```

Keep default `safe` mode for normal smoke tests. The first `text_mlx_lm` run downloads the MLX model and may take several minutes.


In [ ]:
if BACKEND_MODE == "mlx_hybrid" and MLX_ADAPTERS:
    import json
    import subprocess
    import sys

    adapter_map = {
        "text_proof": ["--adapter", "proof"],
        "text_mlx_lm": ["--adapter", "mlx_lm", "--model-name", TEXT_MLX_LM_MODEL],
    }
    for adapter_name in MLX_ADAPTERS:
        if adapter_name not in adapter_map:
            print(f"Unknown MLX adapter {adapter_name!r}; known adapters: {sorted(adapter_map)}")
            continue
        output_path = f"cache_mac_m2/{adapter_name}_report.json"
        print(f"Running MLX adapter: {adapter_name}")
        result = subprocess.run(
            [
                sys.executable,
                "tools/run_mlx_text_proof.py",
                "--json",
                "--output",
                output_path,
                *adapter_map[adapter_name],
            ],
            check=True,
            capture_output=True,
            text=True,
        )
        report = json.loads(result.stdout)
        print(json.dumps(report, indent=2))
else:
    print(
        "MLX-assisted text adapters skipped. Set BACKEND_MODE='mlx_hybrid' and "
        "MLX_ADAPTERS=['text_proof'] or ['text_mlx_lm'] to run them."
    )


## Local video/no-text smoke tests and MLX V-JEPA prediction mode

Default `safe` mode validates local c1/c3 video events without running heavyweight original video extraction.

For the corrected MLX V-JEPA2 path, `mlx_vjepa_predict` runs `tools/run_mlx_vjepa2_full_batch.py`. That batch generates one temporal MLX ViT-g feature artifact per prepared c1/c3 video under `cache_mac_m2_verification/mlx_vjepa_vitg_per_video_temporal/`, then calls `tools/run_mlx_vjepa2_tribe_smoke.py` with `--plot-timesteps 0` so every prediction timestep is visualized. This mode refuses to run unless `TRIBEV2_MLX_VJEPA_PREDICT_OPT_IN=1` and the canonical float32 probe/parity artifacts exist.

The older `TRIBEV2_RUN_LOCAL_VIDEO_PREDICTION=1` path still attempts the original PyTorch/Hugging Face/neuralset extractors and remains separate from the MLX V-JEPA report-backed path.


In [ ]:
from pathlib import Path
import json
import subprocess

import pandas as pd
from neuralset.events.utils import standardize_events

VIDEO_SUFFIXES = {".mp4", ".mov", ".mkv", ".webm", ".avi"}
LOCAL_VIDEO_CASES = {
    "c1_video_audio": {"folder": Path("data/c1_video_audio"), "mode": "video_audio_no_text"},
    "c3_video": {"folder": Path("data/c3_video"), "mode": "video_only_no_text"},
}


def list_video_files(folder: Path) -> list[Path]:
    # Use the smallest files first for a practical Mac smoke test.
    return sorted(
        (p for p in folder.iterdir() if p.suffix.lower() in VIDEO_SUFFIXES),
        key=lambda p: (p.stat().st_size, p.name),
    )


def ffprobe_streams(path: Path) -> tuple[float, bool, bool]:
    cmd = [
        "ffprobe",
        "-v",
        "error",
        "-show_entries",
        "format=duration:stream=codec_type",
        "-of",
        "json",
        str(path),
    ]
    meta = json.loads(subprocess.check_output(cmd, text=True))
    duration = float(meta.get("format", {}).get("duration", 0.0))
    stream_types = {s.get("codec_type") for s in meta.get("streams", [])}
    return duration, "video" in stream_types, "audio" in stream_types


def make_smoke_clip(path: Path, *, keep_audio: bool) -> Path:
    smoke_dir = CACHE_FOLDER / "local_video_smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    suffix = "av" if keep_audio else "video"
    out = smoke_dir / f"{path.stem}_{suffix}_{SMOKE_TEST_SECONDS:g}s_{SMOKE_TEST_WIDTH}w.mp4"
    if out.exists():
        return out
    cmd = [
        "ffmpeg",
        "-y",
        "-i",
        str(path),
        "-t",
        str(SMOKE_TEST_SECONDS),
        "-vf",
        f"scale={SMOKE_TEST_WIDTH}:-2",
        "-r",
        "12",
        "-c:v",
        "libx264",
        "-preset",
        "veryfast",
        "-crf",
        "28",
    ]
    if keep_audio:
        cmd += ["-c:a", "aac", "-b:a", "64k"]
    else:
        cmd += ["-an"]
    cmd.append(str(out))
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return out


def build_video_audio_events_no_text(path: Path) -> pd.DataFrame:
    duration, has_video, has_audio = ffprobe_streams(path)
    if not has_video or not has_audio:
        raise ValueError(f"Expected audio+video streams for {path}; got video={has_video}, audio={has_audio}")
    audio_dir = CACHE_FOLDER / "local_extracted_audio"
    audio_dir.mkdir(parents=True, exist_ok=True)
    audio_path = audio_dir / f"{path.stem}.wav"
    if not audio_path.exists():
        subprocess.run([
            "ffmpeg",
            "-y",
            "-i",
            str(path),
            "-vn",
            "-acodec",
            "pcm_s16le",
            "-ar",
            "44100",
            "-ac",
            "2",
            str(audio_path),
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    events = standardize_events(pd.DataFrame([
        {
            "type": "Video",
            "filepath": str(path),
            "start": 0,
            "duration": duration,
            "timeline": "default",
            "subject": "default",
        },
        {
            "type": "Audio",
            "filepath": str(audio_path),
            "start": 0,
            "duration": duration,
            "timeline": "default",
            "subject": "default",
        },
    ]))
    event_types = sorted(events["type"].dropna().unique().tolist())
    if "Word" in event_types:
        raise AssertionError(f"No-text case unexpectedly produced Word events for {path}")
    return events


def build_video_only_events_no_text(path: Path) -> pd.DataFrame:
    duration, has_video, has_audio = ffprobe_streams(path)
    if not has_video or has_audio:
        raise ValueError(f"Expected video-only stream for {path}; got video={has_video}, audio={has_audio}")
    return standardize_events(pd.DataFrame([
        {
            "type": "Video",
            "filepath": str(path),
            "start": 0,
            "duration": duration,
            "timeline": "default",
            "subject": "default",
        }
    ]))


local_video_event_sets = []
if RUN_LOCAL_VIDEO_DEMO:
    for case_name, cfg in LOCAL_VIDEO_CASES.items():
        files = list_video_files(cfg["folder"])[:MAX_LOCAL_VIDEO_CASES_PER_FOLDER]
        if not files:
            raise FileNotFoundError(f"No video files found in {cfg['folder']}")
        for original_video_path in files:
            original_duration, original_has_video, original_has_audio = ffprobe_streams(original_video_path)
            print(f"{case_name}: source={original_video_path} | duration={original_duration:.2f}s video={original_has_video} audio={original_has_audio}")
            keep_audio = cfg["mode"] == "video_audio_no_text"
            video_path = make_smoke_clip(original_video_path, keep_audio=keep_audio)
            duration, has_video, has_audio = ffprobe_streams(video_path)
            print(f"  smoke clip={video_path} | duration={duration:.2f}s video={has_video} audio={has_audio}")
            if cfg["mode"] == "video_audio_no_text":
                events = build_video_audio_events_no_text(video_path)
            else:
                events = build_video_only_events_no_text(video_path)
            local_video_event_sets.append({
                "case": case_name,
                "source_path": original_video_path,
                "path": video_path,
                "events": events,
                "types": sorted(events["type"].dropna().unique().tolist()),
            })
            display(events.head(8)[["type", "start", "duration", "filepath"]])
            print(f"  event types: {sorted(events['type'].dropna().unique().tolist())}; rows={len(events)}")
else:
    print("RUN_LOCAL_VIDEO_DEMO is False; skipping local video event construction.")


In [ ]:
local_video_predictions = []
local_video_prediction_report = []

if RUN_MLX_VJEPA_PREDICT:
    import subprocess
    import sys

    predict_report_dir = CACHE_FOLDER / "mlx_vjepa_predict_reports"
    batch_output_root = predict_report_dir / "all24_c1_c3_per_video_temporal"
    batch_feature_root = Path("cache_mac_m2_verification/mlx_vjepa_vitg_per_video_temporal")
    required_artifacts = [MLX_VJEPA_FEATURES, MLX_VJEPA_PROBE_REPORT]
    missing = [str(path) for path in required_artifacts if not path.exists()]
    if not MLX_VJEPA_PREDICT_OPT_IN:
        refusal = {
            "status": "refused",
            "reason": "TRIBEV2_MLX_VJEPA_PREDICT_OPT_IN=1 is required for mlx_vjepa_predict",
            "required_artifacts": [str(path) for path in required_artifacts],
            "corrected_batch": "tools/run_mlx_vjepa2_full_batch.py",
        }
        predict_report_dir.mkdir(parents=True, exist_ok=True)
        (predict_report_dir / "refused.json").write_text(json.dumps(refusal, indent=2))
        print(json.dumps(refusal, indent=2))
    elif missing:
        blocker = {"status": "blocked", "reason": "missing_required_artifacts", "missing": missing}
        predict_report_dir.mkdir(parents=True, exist_ok=True)
        (predict_report_dir / "blocked.json").write_text(json.dumps(blocker, indent=2))
        print(json.dumps(blocker, indent=2))
    else:
        cmd = [
            sys.executable,
            "tools/run_mlx_vjepa2_full_batch.py",
            "--feature-root", str(batch_feature_root),
            "--output-root", str(batch_output_root),
            "--cache-folder", str(CACHE_FOLDER / "tribe_smoke_cache_per_video_temporal"),
            "--smoke-timeout-seconds", str(VIDEO_PREDICTION_TIMEOUT_SECONDS),
        ]
        result = subprocess.run(cmd, check=False, capture_output=True, text=True, timeout=(VIDEO_PREDICTION_TIMEOUT_SECONDS + 45) * 24)
        if result.stdout:
            print(result.stdout[-8000:])
        if result.stderr:
            print(result.stderr[-8000:])
        summary_path = batch_output_root / "summary.json"
        report_item = json.loads(summary_path.read_text()) if summary_path.exists() else {
            "status": "failed",
            "returncode": result.returncode,
            "reason": "corrected batch exited without summary.json",
        }
        local_video_prediction_report.append(report_item)
        print(f"Corrected per-video MLX V-JEPA predict batch written to {batch_output_root}")
elif RUN_LOCAL_VIDEO_DEMO and RUN_LOCAL_VIDEO_PREDICTION:

    import subprocess
    import sys

    mode_by_case = {name: cfg["mode"] for name, cfg in LOCAL_VIDEO_CASES.items()}
    for item in local_video_event_sets:
        print(f"\nPredicting {item['case']}: {item['path']}")
        case_report_path = CACHE_FOLDER / "local_video_prediction_cases" / f"{item['case']}.json"
        cmd = [
            sys.executable,
            "tools/run_local_video_prediction_case.py",
            "--case",
            item["case"],
            "--path",
            str(item["path"]),
            "--mode",
            mode_by_case[item["case"]],
            "--cache-folder",
            str(CACHE_FOLDER),
            "--device",
            DEVICE,
            "--output",
            str(case_report_path),
        ]
        if MPS_EXTRACTORS_ENABLED:
            cmd.append("--use-mps-extractors")
        try:
            result = subprocess.run(
                cmd,
                check=False,
                capture_output=True,
                text=True,
                timeout=VIDEO_PREDICTION_TIMEOUT_SECONDS,
            )
            if result.stdout:
                print(result.stdout[-4000:])
            if result.stderr:
                print(result.stderr[-4000:])
            if case_report_path.exists():
                report_item = json.loads(case_report_path.read_text())
            else:
                report_item = {
                    "case": item["case"],
                    "path": str(item["path"]),
                    "event_types": item["types"],
                    "status": "failed",
                    "error_type": "MissingReport",
                    "error": "Prediction subprocess exited without writing a report",
                    "returncode": result.returncode,
                }
        except subprocess.TimeoutExpired as exc:
            report_item = {
                "case": item["case"],
                "path": str(item["path"]),
                "event_types": item["types"],
                "status": "timeout",
                "timeout_seconds": VIDEO_PREDICTION_TIMEOUT_SECONDS,
                "device": DEVICE,
                "mps_extractors_enabled": MPS_EXTRACTORS_ENABLED,
                "error_type": "TimeoutExpired",
                "error": str(exc),
            }
            print(f"Prediction timed out for {item['case']} after {VIDEO_PREDICTION_TIMEOUT_SECONDS}s")
        local_video_prediction_report.append(report_item)
        if report_item.get("status") == "completed":
            print(f"Prediction completed: shape={report_item.get('preds_shape')} segments={report_item.get('segments')}")
        else:
            print(f"Prediction did not complete: {report_item.get('status')} {report_item.get('error_type')} {report_item.get('error', '')[:500]}")
        report_path = CACHE_FOLDER / "local_video_prediction_report.json"
        report_path.write_text(json.dumps(local_video_prediction_report, indent=2))
        print(f"Updated prediction report: {report_path}")
elif RUN_LOCAL_VIDEO_DEMO:
    print(
        "Local video events were validated, but prediction is skipped by default "
        "because TRIBE's V-JEPA2/DINO video feature extraction is still impractically slow on this Mac even with the experimental MPS patch. "
        "Set TRIBEV2_RUN_LOCAL_VIDEO_PREDICTION=1 before starting the kernel only if you want to attempt the full run. "
        f"The attempt will be bounded by TRIBEV2_VIDEO_PREDICTION_TIMEOUT_SECONDS={VIDEO_PREDICTION_TIMEOUT_SECONDS}."
    )
else:
    print("RUN_LOCAL_VIDEO_DEMO is False; skipping local video prediction.")


In [ ]:
if RUN_LOCAL_VIDEO_DEMO and local_video_predictions:
    item = local_video_predictions[0]
    n_timesteps = min(10, len(item["preds"]))
    fig = plotter.plot_timesteps(
        item["preds"][:n_timesteps],
        segments=item["segments"][:n_timesteps],
        cmap="fire",
        norm_percentile=99,
        vmin=.6,
        alpha_cmap=(0, .2),
        show_stimuli=True,
    )
    print(f"Visualized first {n_timesteps} timesteps for {item['case']}: {item['path'].name}")
else:
    print("No local video predictions available to visualize.")


## Cleanup / memory notes

If you run several variants in one kernel, free memory between runs:

```python
for name in ["preds", "segments", "local_video_predictions"]:
    if name in globals():
        del globals()[name]
if DEVICE == "mps":
    torch.mps.empty_cache()
gc.collect()
```

Temporary smoke-test media and extracted audio live under `cache_mac_m2/` and can be deleted safely between runs.

If the kernel is killed during video extraction, restart the kernel, keep `RUN_LOCAL_VIDEO_PREDICTION=False`, and validate the local video event smoke tests first. For full original audio/video inference, a CUDA GPU runtime remains the recommended target unless/until TRIBE gets a native Apple backend.
